# LeetCode 1348 — Tweet Counts Per Frequency bằng Python

Link đề: https://leetcode.com/problems/tweet-counts-per-frequency/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Binary Search, Design, Sorting, Ordered Set

Skeleton LeetCode (bài **thiết kế lớp**):

```python
class TweetCounts:

    def __init__(self):
        ...

    def recordTweet(self, tweetName: str, time: int) -> None:
        ...

    def getTweetCountsPerFrequency(self, freq: str, tweetName: str, startTime: int, endTime: int) -> list[int]:
        ...
```

## 1. Mục tiêu học

1. Đọc hiểu một bài **thiết kế (design)**: nhiều lời gọi phương thức xen kẽ, phải chọn cấu trúc dữ liệu lưu trạng thái.
2. Nắm cách **chia đoạn thời gian** `[startTime, endTime]` thành các khúc (chunk) dài `size` giây: số khúc `= (end − start) // size + 1`, chỉ số khúc của thời điểm `t` là `(t − start) // size`.
3. Đi từ "lưu tất cả rồi quét" → **dict theo tên** → **danh sách đã sắp xếp + `bisect`** → **đếm theo từng giây** (tận dụng `end − start ≤ 10^4`).
4. Biết dùng `bisect.insort`, `bisect_left`, `bisect_right` để đếm số phần tử trong một đoạn.
5. Tự viết **bộ giả lập chuỗi lệnh kiểu LeetCode** để kiểm thử lớp thiết kế.

## 2. Đề bài & ví dụ

Một mạng xã hội muốn thống kê số tweet theo từng khoảng thời gian. Một khoảng `[startTime, endTime]` (giây, **bao gồm cả hai đầu**) được chia thành các khúc theo tần suất:

- `"minute"`: khúc 60 giây; `"hour"`: khúc 3600 giây; `"day"`: khúc 86400 giây.
- Ví dụ khoảng `[10, 10000]` theo phút: `[10,69], [70,129], ..., [9970,10000]` — **khúc cuối có thể ngắn hơn** và luôn kết thúc tại `endTime`.

Cài đặt lớp `TweetCounts`:
- `TweetCounts()` khởi tạo.
- `recordTweet(tweetName, time)` ghi nhận một tweet tên `tweetName` tại thời điểm `time`.
- `getTweetCountsPerFrequency(freq, tweetName, startTime, endTime)` trả về list số tweet tên `tweetName` trong **từng khúc** của `[startTime, endTime]`.

**Ví dụ:**
```
["TweetCounts","recordTweet","recordTweet","recordTweet","getTweetCountsPerFrequency","getTweetCountsPerFrequency","recordTweet","getTweetCountsPerFrequency"]
[[],["tweet3",0],["tweet3",60],["tweet3",10],["minute","tweet3",0,59],["minute","tweet3",0,60],["tweet3",120],["hour","tweet3",0,210]]
Output: [null,null,null,null,[2],[2,1],null,[4]]
```

**Ràng buộc:** `0 <= time, startTime, endTime <= 10^9`; `0 <= endTime − startTime <= 10^4`; tối đa `10^4` lời gọi tổng cộng.

In [1]:
import bisect
import random
import time as _time
from collections import defaultdict, Counter

EX_OPS = ["TweetCounts", "recordTweet", "recordTweet", "recordTweet", "getTweetCountsPerFrequency",
          "getTweetCountsPerFrequency", "recordTweet", "getTweetCountsPerFrequency"]
EX_ARGS = [[], ["tweet3", 0], ["tweet3", 60], ["tweet3", 10], ["minute", "tweet3", 0, 59],
           ["minute", "tweet3", 0, 60], ["tweet3", 120], ["hour", "tweet3", 0, 210]]
EX_OUT = [None, None, None, None, [2], [2, 1], None, [4]]

def run(cls, ops, args):
    """Bộ GIẢ LẬP kiểu LeetCode: ops[0] là tên lớp, các ops sau là tên phương thức."""
    obj = cls(*args[0])
    out = [None]
    for op, a in zip(ops[1:], args[1:]):
        out.append(getattr(obj, op)(*a))
    return out

print("Số lệnh trong ví dụ:", len(EX_OPS), "| output mong đợi:", EX_OUT)

Số lệnh trong ví dụ: 8 | output mong đợi: [None, None, None, None, [2], [2, 1], None, [4]]


## 3. Kiến thức nền

### 3.1 Bài thiết kế lớp & bộ giả lập

LeetCode gọi `obj = TweetCounts()` rồi lần lượt gọi các phương thức theo hai mảng `ops`, `args`. Hàm `run` ở trên là **bản giả lập** cơ chế đó: dùng `getattr(obj, tên)` để lấy phương thức theo tên chuỗi rồi gọi với `*a`.

In [2]:
class Demo:
    def __init__(self):
        self.x = 0
    def add(self, v):
        self.x += v
    def get(self):
        return self.x

print(run(Demo, ["Demo", "add", "add", "get"], [[], [3], [4], []]))

[None, None, None, 7]


### 3.2 Chia khúc thời gian bằng phép chia nguyên

Với khúc dài `size`, thời điểm `t ∈ [start, end]` thuộc khúc số `(t − start) // size` (đánh số từ 0). Số khúc là `(end − start) // size + 1` — **luôn ≥ 1** kể cả khi `start == end`.

In [3]:
start, end, size = 10, 10000, 60
n_chunks = (end - start) // size + 1
print("số khúc:", n_chunks)
for k in [0, 1, 2, n_chunks - 1]:
    lo = start + k * size
    hi = min(lo + size - 1, end)
    print(f"khúc {k}: [{lo}, {hi}]")
for t in [10, 69, 70, 10000]:
    print(f"t={t} -> khúc {(t - start) // size}")

số khúc: 167
khúc 0: [10, 69]
khúc 1: [70, 129]
khúc 2: [130, 189]
khúc 166: [9970, 10000]
t=10 -> khúc 0
t=69 -> khúc 0
t=70 -> khúc 1
t=10000 -> khúc 166


### 3.3 Từ điển ánh xạ tần suất → số giây

Dùng một dict hằng số thay cho chuỗi `if/elif`.

In [4]:
FREQ = {"minute": 60, "hour": 3600, "day": 86400}
print(FREQ["hour"], FREQ["day"] // FREQ["hour"])

3600 24


### 3.4 `defaultdict(list)` — nhóm theo tên

`defaultdict(list)` tự tạo list rỗng khi gặp khóa mới, tiện cho việc gom các thời điểm theo `tweetName`.

In [5]:
d = defaultdict(list)
d["tweet3"].append(0)
d["tweet3"].append(60)
d["tweetA"].append(5)
print(dict(d), "| khóa chưa có:", d["nobody"])

{'tweet3': [0, 60], 'tweetA': [5]} | khóa chưa có: []


### 3.5 `bisect` — tìm kiếm nhị phân trên list đã sắp xếp

- `bisect.insort(a, x)`: chèn `x` vào đúng vị trí để `a` vẫn sắp xếp.
- `bisect_left(a, x)`: vị trí đầu tiên có `a[i] >= x`.
- `bisect_right(a, x)`: vị trí đầu tiên có `a[i] > x`.
- Số phần tử trong đoạn `[lo, hi]` = `bisect_right(a, hi) − bisect_left(a, lo)`.

In [6]:
a = []
for x in [60, 0, 10, 120, 60]:
    bisect.insort(a, x)
print("a =", a)
lo, hi = 0, 59
print(f"số phần tử trong [{lo},{hi}] =", bisect.bisect_right(a, hi) - bisect.bisect_left(a, lo))
print("bisect_left(a, 60) =", bisect.bisect_left(a, 60), "| bisect_right(a, 60) =", bisect.bisect_right(a, 60))

a = [0, 10, 60, 60, 120]
số phần tử trong [0,59] = 2
bisect_left(a, 60) = 2 | bisect_right(a, 60) = 4


### 3.6 `Counter` — đếm số tweet tại từng giây

Nhiều tweet có thể ở **cùng một thời điểm** — phải đếm đủ, không được dùng `set`.

In [7]:
c = Counter()
for t in [5, 5, 7]:
    c[t] += 1
print(c, "| tại giây 6:", c[6])

Counter({5: 2, 7: 1}) | tại giây 6: 0


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ: các tweet `"tweet3"` ở thời điểm `0, 60, 10` (rồi thêm `120`).

1. `("minute", 0, 59)`: 1 khúc `[0,59]`; các tweet `0, 10` thuộc khúc 0 ⇒ `[2]`.
2. `("minute", 0, 60)`: `(60−0)//60 + 1 = 2` khúc `[0,59]`, `[60,60]`; `0,10` → khúc 0, `60` → khúc 1 ⇒ `[2, 1]`.
3. Thêm `120`. `("hour", 0, 210)`: `210//3600 + 1 = 1` khúc `[0,210]`; cả `0, 10, 60, 120` ⇒ `[4]`.

| Truy vấn | size | số khúc | tweet → khúc | Kết quả |
|---|---|---|---|---|
| minute 0..59 | 60 | 1 | 0→0, 10→0, (60 ngoài) | [2] |
| minute 0..60 | 60 | 2 | 0→0, 10→0, 60→1 | [2, 1] |
| hour 0..210 | 3600 | 1 | 0,10,60,120 → 0 | [4] |

**Trực giác:** mỗi truy vấn = "lọc các tweet của tên đó nằm trong `[start, end]`, rồi bỏ mỗi tweet vào khúc `(t − start) // size`".

In [8]:
def answer_by_hand(times, size, start, end):
    res = [0] * ((end - start) // size + 1)
    for t in times:
        if start <= t <= end:
            res[(t - start) // size] += 1
    return res

print(answer_by_hand([0, 60, 10], 60, 0, 59))
print(answer_by_hand([0, 60, 10], 60, 0, 60))
print(answer_by_hand([0, 60, 10, 120], 3600, 0, 210))

[2]
[2, 1]
[4]


## 5. Cách 1 — Brute Force / baseline: lưu mọi tweet vào một list

**Ý tưởng:** `recordTweet` chỉ `append((tên, thời điểm))` vào một list chung. Mỗi truy vấn quét **toàn bộ** list, lọc theo tên và đoạn thời gian.

In [9]:
class TweetCountsBrute:
    FREQ = {"minute": 60, "hour": 3600, "day": 86400}

    def __init__(self):
        self.all = []                       # list (tên, thời điểm)

    def recordTweet(self, tweetName, time):
        self.all.append((tweetName, time))

    def getTweetCountsPerFrequency(self, freq, tweetName, startTime, endTime):
        size = self.FREQ[freq]
        res = [0] * ((endTime - startTime) // size + 1)
        for name, t in self.all:            # quét hết mọi tweet
            if name == tweetName and startTime <= t <= endTime:
                res[(t - startTime) // size] += 1
        return res

got = run(TweetCountsBrute, EX_OPS, EX_ARGS)
print(got, "| đúng?", got == EX_OUT)

[None, None, None, None, [2], [2, 1], None, [4]] | đúng? True


**Độ phức tạp:** `recordTweet` `O(1)`; truy vấn `O(N + K)` với `N` = tổng số tweet đã ghi (mọi tên), `K` = số khúc. Bộ nhớ `O(N)`.

**Nút thắt:** truy vấn phải duyệt cả các tweet **khác tên** và các tweet **ngoài khoảng thời gian**. Với `10^4` lời gọi, xấu nhất ~`5000 × 5000 = 2,5·10^7` phép so sánh — vẫn qua được nhưng lãng phí.

## 6. Ý tưởng thuật toán / bước đột phá

Hai bước cải tiến độc lập:

1. **Nhóm theo tên** bằng dict: truy vấn chỉ nhìn các tweet đúng tên ⇒ loại bỏ phần "khác tên".
2. **Sắp xếp thời điểm** của mỗi tên: các tweet trong `[start, end]` nằm **liên tiếp** trong list ⇒ dùng `bisect` nhảy thẳng tới đoạn đó ⇒ loại bỏ phần "ngoài khoảng".

**Quy tắc nếu… thì…:**
- Nếu `t < start` hoặc `t > end` thì bỏ qua; ngược lại cộng vào khúc `(t − start) // size`.
- Nếu list thời điểm đã sắp xếp thì đoạn hợp lệ là `a[bisect_left(a, start) : bisect_right(a, end)]`.

**Một quan sát khác:** `end − start ≤ 10^4` ⇒ có thể duyệt **từng giây** trong khoảng, tra `Counter` số tweet tại giây đó — chi phí không phụ thuộc số tweet.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Dict theo tên, list chưa sắp xếp

**Giải thích:** `self.t[tên]` là list thời điểm. Truy vấn chỉ quét list của tên đó.

In [10]:
class TweetCountsDict:
    FREQ = {"minute": 60, "hour": 3600, "day": 86400}

    def __init__(self):
        self.t = defaultdict(list)

    def recordTweet(self, tweetName, time):
        self.t[tweetName].append(time)

    def getTweetCountsPerFrequency(self, freq, tweetName, startTime, endTime):
        size = self.FREQ[freq]
        res = [0] * ((endTime - startTime) // size + 1)
        for x in self.t[tweetName]:
            if startTime <= x <= endTime:
                res[(x - startTime) // size] += 1
        return res

print(run(TweetCountsDict, EX_OPS, EX_ARGS) == EX_OUT)

True


Code nộp LeetCode (Cách 2):

In [11]:
class TweetCounts:

    def __init__(self):
        self.t = defaultdict(list)
        self.size = {"minute": 60, "hour": 3600, "day": 86400}

    def recordTweet(self, tweetName: str, time: int) -> None:
        self.t[tweetName].append(time)

    def getTweetCountsPerFrequency(self, freq: str, tweetName: str, startTime: int, endTime: int) -> list[int]:
        size = self.size[freq]
        res = [0] * ((endTime - startTime) // size + 1)
        for x in self.t[tweetName]:
            if startTime <= x <= endTime:
                res[(x - startTime) // size] += 1
        return res

print(run(TweetCounts, EX_OPS, EX_ARGS))

[None, None, None, None, [2], [2, 1], None, [4]]


**Độ phức tạp:** record `O(1)`; truy vấn `O(k + K)` với `k` = số tweet của tên đó.

### 7.2 Cách 3 — Dict theo tên, list **đã sắp xếp** + `bisect`

**Giải thích từng đoạn:**
1. `recordTweet`: `bisect.insort(self.t[tên], time)` giữ list luôn tăng dần (chèn `O(k)` do dời phần tử, nhưng rất nhanh trong thực tế vì là `memmove`).
2. Truy vấn: `i = bisect_left(a, start)`, `j = bisect_right(a, end)` ⇒ chỉ duyệt `a[i:j]`.
3. Mỗi phần tử trong đoạn → khúc `(x − start) // size`.

Biến thể: với mỗi khúc `[lo, hi]` gọi hai lần `bisect` ⇒ `O(K log k)` — tốt khi số tweet trong khoảng rất lớn còn số khúc nhỏ.

In [12]:
class TweetCountsBisect:
    FREQ = {"minute": 60, "hour": 3600, "day": 86400}

    def __init__(self):
        self.t = defaultdict(list)

    def recordTweet(self, tweetName, time):
        bisect.insort(self.t[tweetName], time)

    def getTweetCountsPerFrequency(self, freq, tweetName, startTime, endTime):
        size = self.FREQ[freq]
        a = self.t[tweetName]
        res = [0] * ((endTime - startTime) // size + 1)
        i = bisect.bisect_left(a, startTime)
        j = bisect.bisect_right(a, endTime)
        for x in a[i:j]:                     # chỉ các tweet trong khoảng
            res[(x - startTime) // size] += 1
        return res

class TweetCountsBisectChunk(TweetCountsBisect):
    """Biến thể: mỗi khúc đếm bằng 2 lần bisect."""
    def getTweetCountsPerFrequency(self, freq, tweetName, startTime, endTime):
        size = self.FREQ[freq]
        a = self.t[tweetName]
        res = []
        lo = startTime
        while lo <= endTime:
            hi = min(lo + size - 1, endTime)
            res.append(bisect.bisect_right(a, hi) - bisect.bisect_left(a, lo))
            lo += size
        return res

print(run(TweetCountsBisect, EX_OPS, EX_ARGS) == EX_OUT, run(TweetCountsBisectChunk, EX_OPS, EX_ARGS) == EX_OUT)

True True


Code nộp LeetCode (Cách 3):

In [13]:
class TweetCounts:

    def __init__(self):
        self.t = defaultdict(list)
        self.size = {"minute": 60, "hour": 3600, "day": 86400}

    def recordTweet(self, tweetName: str, time: int) -> None:
        bisect.insort(self.t[tweetName], time)

    def getTweetCountsPerFrequency(self, freq: str, tweetName: str, startTime: int, endTime: int) -> list[int]:
        size = self.size[freq]
        a = self.t[tweetName]
        res = [0] * ((endTime - startTime) // size + 1)
        i = bisect.bisect_left(a, startTime)
        j = bisect.bisect_right(a, endTime)
        for x in a[i:j]:
            res[(x - startTime) // size] += 1
        return res

print(run(TweetCounts, EX_OPS, EX_ARGS))

[None, None, None, None, [2], [2, 1], None, [4]]


**Độ phức tạp:** record `O(k)` (chèn list) hoặc `O(log k)` nếu dùng cây cân bằng / `SortedList` (thư viện ngoài, không dùng ở đây); truy vấn `O(log k + m + K)` với `m` = số tweet trong khoảng.

### 7.3 Cách 4 — Đếm theo từng giây (`Counter` theo tên)

**Giải thích:** `self.c[tên][giây]` = số tweet tại giây đó. Truy vấn duyệt `t` từ `start` đến `end` (≤ `10^4 + 1` giá trị nhờ ràng buộc) và cộng `c[t]` vào khúc `(t − start) // size`. Không cần sắp xếp; chi phí truy vấn chỉ phụ thuộc độ dài khoảng.

In [14]:
class TweetCountsPerSecond:
    FREQ = {"minute": 60, "hour": 3600, "day": 86400}

    def __init__(self):
        self.c = defaultdict(Counter)

    def recordTweet(self, tweetName, time):
        self.c[tweetName][time] += 1

    def getTweetCountsPerFrequency(self, freq, tweetName, startTime, endTime):
        size = self.FREQ[freq]
        cnt = self.c[tweetName]
        res = [0] * ((endTime - startTime) // size + 1)
        if len(cnt) <= endTime - startTime:          # ít thời điểm khác nhau -> duyệt các khóa
            for x, v in cnt.items():
                if startTime <= x <= endTime:
                    res[(x - startTime) // size] += v
        else:                                        # nhiều -> duyệt từng giây
            for x in range(startTime, endTime + 1):
                res[(x - startTime) // size] += cnt.get(x, 0)
        return res

print(run(TweetCountsPerSecond, EX_OPS, EX_ARGS) == EX_OUT)

True


**Độ phức tạp:** record `O(1)`; truy vấn `O(min(d, L) + K)` với `d` = số thời điểm khác nhau của tên, `L = end − start + 1 ≤ 10^4 + 1`. Bộ nhớ `O(d)`.

Lưu ý: `cnt.get(x, 0)` thay vì `cnt[x]` để **không tạo khóa mới** trong Counter khi duyệt (tránh phình bộ nhớ).

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ của đề, in trạng thái list đã sắp xếp và các chỉ số `bisect`.

| Lệnh | list `a` của tweet3 | i | j | Kết quả |
|---|---|---|---|---|
| record 0 | [0] | | | null |
| record 60 | [0, 60] | | | null |
| record 10 | [0, 10, 60] | | | null |
| minute 0..59 | [0, 10, 60] | 0 | 2 | [2] |
| minute 0..60 | [0, 10, 60] | 0 | 3 | [2, 1] |
| record 120 | [0, 10, 60, 120] | | | null |
| hour 0..210 | [0, 10, 60, 120] | 0 | 4 | [4] |

In [15]:
class TweetCountsTrace(TweetCountsBisect):
    def recordTweet(self, tweetName, time):
        super().recordTweet(tweetName, time)
        print(f"record({tweetName!r}, {time}) -> a = {self.t[tweetName]}")

    def getTweetCountsPerFrequency(self, freq, tweetName, startTime, endTime):
        size = self.FREQ[freq]
        a = self.t[tweetName]
        i, j = bisect.bisect_left(a, startTime), bisect.bisect_right(a, endTime)
        res = [0] * ((endTime - startTime) // size + 1)
        print(f"query({freq}, {startTime}..{endTime}): size={size}, số khúc={len(res)}, i={i}, j={j}, đoạn={a[i:j]}")
        for x in a[i:j]:
            k = (x - startTime) // size
            res[k] += 1
            print(f"    tweet {x} -> khúc {k} -> res={res}")
        return res

def trace_run(ops, args):
    return run(TweetCountsTrace, ops, args)

print("Kết quả:", trace_run(EX_OPS, EX_ARGS))

record('tweet3', 0) -> a = [0]
record('tweet3', 60) -> a = [0, 60]
record('tweet3', 10) -> a = [0, 10, 60]
query(minute, 0..59): size=60, số khúc=1, i=0, j=2, đoạn=[0, 10]
    tweet 0 -> khúc 0 -> res=[1]
    tweet 10 -> khúc 0 -> res=[2]
query(minute, 0..60): size=60, số khúc=2, i=0, j=3, đoạn=[0, 10, 60]
    tweet 0 -> khúc 0 -> res=[1, 0]
    tweet 10 -> khúc 0 -> res=[2, 0]
    tweet 60 -> khúc 1 -> res=[2, 1]
record('tweet3', 120) -> a = [0, 10, 60, 120]
query(hour, 0..210): size=3600, số khúc=1, i=0, j=4, đoạn=[0, 10, 60, 120]
    tweet 0 -> khúc 0 -> res=[1]
    tweet 10 -> khúc 0 -> res=[2]
    tweet 60 -> khúc 0 -> res=[3]
    tweet 120 -> khúc 0 -> res=[4]
Kết quả: [None, None, None, None, [2], [2, 1], None, [4]]


Trace thêm một truy vấn nhiều khúc: các tweet tại `5, 65, 65, 130, 200`, truy vấn `("minute", 0, 199)`.

In [16]:
ops = ["TweetCounts"] + ["recordTweet"] * 5 + ["getTweetCountsPerFrequency"]
args = [[]] + [["x", t] for t in [5, 65, 65, 130, 200]] + [["minute", "x", 0, 199]]
print("Kết quả:", trace_run(ops, args))

record('x', 5) -> a = [5]
record('x', 65) -> a = [5, 65]
record('x', 65) -> a = [5, 65, 65]
record('x', 130) -> a = [5, 65, 65, 130]
record('x', 200) -> a = [5, 65, 65, 130, 200]
query(minute, 0..199): size=60, số khúc=4, i=0, j=4, đoạn=[5, 65, 65, 130]
    tweet 5 -> khúc 0 -> res=[1, 0, 0, 0]
    tweet 65 -> khúc 1 -> res=[1, 1, 0, 0]
    tweet 65 -> khúc 1 -> res=[1, 2, 0, 0]
    tweet 130 -> khúc 2 -> res=[1, 2, 1, 0]
Kết quả: [None, None, None, None, None, None, [1, 2, 1, 0]]


## 9. Code nộp LeetCode

Chọn Cách 3 (dict + list sắp xếp + `bisect`) — cân bằng giữa đơn giản và hiệu quả, đúng chữ ký đề.

In [17]:
import bisect
from collections import defaultdict

class TweetCounts:

    def __init__(self):
        self.t = defaultdict(list)                       # tên -> list thời điểm tăng dần
        self.size = {"minute": 60, "hour": 3600, "day": 86400}

    def recordTweet(self, tweetName: str, time: int) -> None:
        bisect.insort(self.t[tweetName], time)

    def getTweetCountsPerFrequency(self, freq: str, tweetName: str, startTime: int, endTime: int) -> list[int]:
        size = self.size[freq]
        a = self.t[tweetName]
        res = [0] * ((endTime - startTime) // size + 1)
        i = bisect.bisect_left(a, startTime)
        j = bisect.bisect_right(a, endTime)
        for x in a[i:j]:
            res[(x - startTime) // size] += 1
        return res

got = run(TweetCounts, EX_OPS, EX_ARGS)
assert got == EX_OUT
print(got)

[None, None, None, None, [2], [2, 1], None, [4]]


## 10. Vì sao thuật toán đúng

1. **Phân hoạch khoảng:** khúc thứ `k` là `[start + k·size, min(start + (k+1)·size − 1, end)]`. Một số nguyên `t ∈ [start, end]` thuộc khúc `k` ⇔ `k·size ≤ t − start < (k+1)·size` ⇔ `k = (t − start) // size`. Khúc cuối có chỉ số `(end − start) // size` ⇒ số khúc `(end − start)//size + 1`. Mỗi `t` thuộc **đúng một** khúc.
2. **Bất biến của Cách 3:** sau mỗi `recordTweet`, `self.t[tên]` chứa **đúng** multiset các thời điểm đã ghi của tên đó, sắp tăng dần (`insort` giữ thứ tự, lưu cả trùng lặp).
3. **Đoạn bisect:** trên list tăng dần, các phần tử `≥ start` bắt đầu từ `bisect_left(a, start)`, các phần tử `≤ end` kết thúc trước `bisect_right(a, end)` ⇒ `a[i:j]` là đúng các tweet trong `[start, end]`.
4. Kết hợp 1–3: mỗi tweet hợp lệ được cộng đúng một lần vào đúng khúc; tweet không hợp lệ không được cộng ⇒ kết quả đúng.

## 11. Phân tích Time / Space

Ký hiệu: `N` = tổng số tweet đã ghi, `k` = số tweet của tên được hỏi, `m` = số tweet của tên đó trong `[start, end]`, `K = (end − start)//size + 1` (≤ 10^4 + 1), `L = end − start + 1`.

| Cách | recordTweet | getTweetCounts | Space |
|---|---|---|---|
| 1. Một list chung | `O(1)` | `O(N + K)` | `O(N)` |
| 2. Dict + list | `O(1)` | `O(k + K)` | `O(N)` |
| 3. Dict + list sắp xếp + bisect | `O(k)` chèn (worst), thường rất nhanh | `O(log k + m + K)` | `O(N)` |
| 3b. bisect từng khúc | `O(k)` | `O(K log k)` | `O(N)` |
| 4. Counter theo giây | `O(1)` | `O(min(d, L) + K)` | `O(N)` |

Với giới hạn `10^4` lời gọi, mọi cách đều qua; Cách 3/4 tốt hơn rõ khi nhiều tên hoặc nhiều truy vấn.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case (dùng bộ giả lập)

In [18]:
IMPLS = [TweetCountsBrute, TweetCountsDict, TweetCountsBisect, TweetCountsBisectChunk, TweetCountsPerSecond, TweetCounts]

edge_ops = ["TweetCounts", "recordTweet", "recordTweet", "recordTweet", "recordTweet",
            "getTweetCountsPerFrequency", "getTweetCountsPerFrequency", "getTweetCountsPerFrequency",
            "getTweetCountsPerFrequency", "getTweetCountsPerFrequency", "getTweetCountsPerFrequency"]
edge_args = [[], ["a", 5], ["a", 5], ["b", 5], ["a", 10**9],
             ["minute", "a", 5, 5],          # start == end -> 1 khúc, tweet trùng giờ đếm 2
             ["minute", "zzz", 0, 100],      # tên chưa từng xuất hiện -> toàn 0
             ["day", "a", 0, 10000],         # 1 khúc
             ["minute", "a", 6, 120],        # không có tweet trong khoảng
             ["hour", "a", 10**9 - 10000, 10**9],  # thời điểm lớn
             ["minute", "a", 0, 60]]         # tweet ở khúc 0, khúc 1 trống
edge_out = [None, None, None, None, None, [2], [0, 0], [2], [0, 0], [0, 0, 1], [2, 0]]
for cls in IMPLS:
    assert run(cls, EX_OPS, EX_ARGS) == EX_OUT, cls.__name__
    assert run(cls, edge_ops, edge_args) == edge_out, (cls.__name__, run(cls, edge_ops, edge_args))
print("Ví dụ đề + edge case: mọi cài đặt đều đúng")

Ví dụ đề + edge case: mọi cài đặt đều đúng


### 12.2 Đối chiếu ngẫu nhiên (seed cố định, đúng ràng buộc)

Sinh chuỗi lệnh ngẫu nhiên: vài tên, thời điểm tập trung trong một vùng nhỏ để có nhiều trùng lặp, `end − start ≤ 10^4`. So mọi cài đặt với Cách 1.

In [19]:
def random_ops(rng, n_calls):
    names = ["t1", "t2", "t3"]
    base = rng.choice([0, 10**6, 10**9 - 20000])
    ops, args = ["TweetCounts"], [[]]
    for _ in range(n_calls):
        if rng.random() < 0.6:
            ops.append("recordTweet")
            args.append([rng.choice(names), base + rng.randint(0, 15000)])
        else:
            s = base + rng.randint(0, 12000)
            e = s + rng.randint(0, 10**4)
            e = min(e, 10**9)
            ops.append("getTweetCountsPerFrequency")
            args.append([rng.choice(["minute", "hour", "day"]), rng.choice(names), s, e])
    return ops, args

rng = random.Random(1348)
for _ in range(150):
    ops, args = random_ops(rng, rng.randint(1, 60))
    ref = run(TweetCountsBrute, ops, args)
    for cls in IMPLS[1:]:
        assert run(cls, ops, args) == ref, cls.__name__
print("150 chuỗi lệnh ngẫu nhiên: mọi cài đặt khớp brute force")

150 chuỗi lệnh ngẫu nhiên: mọi cài đặt khớp brute force


### 12.3 Kiểm tra tính chất + thời gian ở giới hạn

Tính chất: tổng các phần tử kết quả = số tweet của tên đó trong `[start, end]` và độ dài kết quả = `(end − start)//size + 1`.

In [20]:
rng = random.Random(99)
ops, args = ["TweetCounts"], [[]]
for _ in range(5000):
    ops.append("recordTweet"); args.append(["hot", rng.randint(0, 10**4)])
for _ in range(5000):
    s = rng.randint(0, 5000)
    ops.append("getTweetCountsPerFrequency"); args.append(["minute", "hot", s, s + 10**4])
for cls in [TweetCountsDict, TweetCountsBisect, TweetCountsPerSecond]:
    st = _time.perf_counter()
    out = run(cls, ops, args)
    print(f"{cls.__name__:22} {_time.perf_counter() - st:.3f}s")
times = sorted(a[1] for a in args[1:5001])
for (freq, name, s, e), res in zip(args[5001:5011], out[5001:5011]):
    assert len(res) == (e - s) // 60 + 1
    assert sum(res) == bisect.bisect_right(times, e) - bisect.bisect_left(times, s)
print("Tính chất tổng & độ dài đúng")

TweetCountsDict        1.741s


TweetCountsBisect      0.949s


TweetCountsPerSecond   1.426s
Tính chất tổng & độ dài đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Thiếu `+ 1` khi tính số khúc

In [21]:
class WrongChunks(TweetCountsDict):
    def getTweetCountsPerFrequency(self, freq, tweetName, startTime, endTime):
        size = self.FREQ[freq]
        res = [0] * ((endTime - startTime) // size)        # thiếu +1
        for x in self.t[tweetName]:
            if startTime <= x <= endTime:
                k = (x - startTime) // size
                if k < len(res):                           # tránh IndexError để thấy kết quả sai
                    res[k] += 1
        return res

print("sai :", run(WrongChunks, EX_OPS, EX_ARGS))
print("đúng:", EX_OUT)

sai : [None, None, None, None, [], [2], None, []]
đúng: [None, None, None, None, [2], [2, 1], None, [4]]


**Vì sao sai:** `[0, 59]` có `59 // 60 = 0` ⇒ list rỗng, trong khi luôn có ít nhất 1 khúc. **Sửa:** `(end − start) // size + 1`.

### Lỗi 2 — Tính khúc bằng `t // size` thay vì `(t − start) // size`

In [22]:
class WrongIndex(TweetCountsDict):
    def getTweetCountsPerFrequency(self, freq, tweetName, startTime, endTime):
        size = self.FREQ[freq]
        res = [0] * ((endTime - startTime) // size + 1)
        for x in self.t[tweetName]:
            if startTime <= x <= endTime:
                k = x // size                               # không trừ startTime
                if k < len(res):
                    res[k] += 1
        return res

ops = ["TweetCounts", "recordTweet", "recordTweet", "getTweetCountsPerFrequency"]
args = [[], ["a", 50], ["a", 70], ["minute", "a", 30, 100]]
print("sai :", run(WrongIndex, ops, args))
print("đúng:", run(TweetCounts, ops, args))

sai : [None, None, None, [1, 1]]
đúng: [None, None, None, [2, 0]]


**Vì sao sai:** khúc được neo tại `startTime` (`[30,89]`, `[90,100]`), không neo tại bội của 60. **Sửa:** `k = (x − startTime) // size`.

### Lỗi 3 — Dùng `set` lưu thời điểm (mất tweet trùng giờ)

In [23]:
class WrongSet(TweetCountsDict):
    def __init__(self):
        self.t = defaultdict(set)
    def recordTweet(self, tweetName, time):
        self.t[tweetName].add(time)

ops = ["TweetCounts", "recordTweet", "recordTweet", "getTweetCountsPerFrequency"]
args = [[], ["a", 5], ["a", 5], ["minute", "a", 0, 59]]
print("sai :", run(WrongSet, ops, args))
print("đúng:", run(TweetCounts, ops, args))

sai : [None, None, None, [1]]
đúng: [None, None, None, [2]]


**Vì sao sai:** hai tweet cùng tên cùng giây là **hai tweet**. **Sửa:** dùng list (hoặc Counter).

### Lỗi 4 — Dùng `bisect_left` cho cận phải (bỏ sót tweet đúng tại `endTime`)

In [24]:
class WrongBisect(TweetCountsBisect):
    def getTweetCountsPerFrequency(self, freq, tweetName, startTime, endTime):
        size = self.FREQ[freq]
        a = self.t[tweetName]
        res = [0] * ((endTime - startTime) // size + 1)
        for x in a[bisect.bisect_left(a, startTime):bisect.bisect_left(a, endTime)]:   # sai cận phải
            res[(x - startTime) // size] += 1
        return res

print("sai :", run(WrongBisect, EX_OPS, EX_ARGS))
print("đúng:", EX_OUT)

sai : [None, None, None, None, [2], [2, 0], None, [4]]
đúng: [None, None, None, None, [2], [2, 1], None, [4]]


**Vì sao sai:** đoạn `[start, end]` **bao gồm** `end`; `bisect_left(a, end)` dừng **trước** các phần tử bằng `end` (tweet tại 60 trong truy vấn `0..60` bị mất). **Sửa:** cận phải dùng `bisect_right(a, endTime)`.

### Lỗi 5 — Dùng `cnt[x]` khi duyệt từng giây (Counter phình ra)

In [25]:
cnt = Counter({5: 1})
for x in range(0, 10001):
    _ = cnt[x]                 # Counter KHÔNG thêm khóa khi đọc...
print("Counter sau khi đọc:", len(cnt))
dd = defaultdict(int, {5: 1})
for x in range(0, 10001):
    _ = dd[x]                  # ...nhưng defaultdict THÌ CÓ
print("defaultdict sau khi đọc:", len(dd), "khóa -> bộ nhớ phình to qua mỗi truy vấn")

Counter sau khi đọc: 1
defaultdict sau khi đọc: 10001 khóa -> bộ nhớ phình to qua mỗi truy vấn


**Vì sao sai:** với `defaultdict(int)`, mỗi lần đọc `dd[x]` tạo khóa mới ⇒ sau nhiều truy vấn, bộ nhớ và thời gian duyệt khóa tăng mạnh. **Sửa:** dùng `.get(x, 0)` (an toàn cho mọi loại dict) — như Cách 4.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Bài design có "đếm trong khoảng thời gian"** ⇒ dict theo khóa + list thời điểm (sắp xếp) + `bisect` — pattern giống 981 (Time Based Key-Value Store), 362 (Design Hit Counter), 2034.
- **Chia khúc**: luôn viết `n = (end − start)//size + 1` và `k = (t − start)//size`; thử ngay ca `start == end`.
- Đọc kỹ ràng buộc: `end − start ≤ 10^4` cho phép duyệt từng giây; `time ≤ 10^9` cấm tạo mảng theo thời gian tuyệt đối.
- Viết **bộ giả lập `run(cls, ops, args)`** trước — dùng lại cho mọi bài design.
- Đoạn đóng `[lo, hi]` trên list sắp xếp: `bisect_right(hi) − bisect_left(lo)`.

## 15. Biến thể của bài

1. **Tần suất tùy ý** (truyền `size` bất kỳ thay vì 3 chuỗi) — chỉ cần đổi dict `FREQ`.
2. **Xóa tweet** `deleteTweet(name, time)`: với list sắp xếp, `bisect_left` tìm rồi `pop` (Bài 5).
3. **Đếm mọi tên** trong khoảng: thêm một list chung "tất cả".
4. **LeetCode 362 — Design Hit Counter**: đếm số hit trong 300 giây gần nhất (hàng đợi / mảng vòng).
5. **Truy vấn rất nhiều khúc, rất nhiều tweet**: dùng prefix sum trên mảng đếm theo giây trong khoảng, hoặc cây Fenwick nếu thời gian được nén tọa độ.

## 16. So sánh các cách

| Cách | Ý tưởng | record | query | Khi nào dùng |
|---|---|---|---|---|
| 1. Một list chung | quét hết | `O(1)` | `O(N + K)` | baseline, làm trọng tài |
| 2. Dict + list | lọc theo tên | `O(1)` | `O(k + K)` | đơn giản, đủ qua LeetCode |
| 3. Dict + sorted list + bisect | nhảy tới đoạn hợp lệ | `O(k)` (memmove) | `O(log k + m + K)` | nhiều tweet ngoài khoảng |
| 3b. bisect mỗi khúc | 2 bisect / khúc | `O(k)` | `O(K log k)` | ít khúc, cực nhiều tweet trong khoảng |
| 4. Counter theo giây | duyệt theo thời gian | `O(1)` | `O(min(d, L) + K)` | khoảng ngắn, nhiều tweet trùng giờ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** Khoảng `[10, 10000]` theo `"hour"` có mấy khúc? Khúc cuối là gì? Thời điểm `7210` thuộc khúc số mấy?

**Bài 2 (tính tay):** Tweet `"x"` tại `1, 59, 60, 61, 119, 120, 3599`. Tính kết quả `("minute", "x", 1, 121)` và `("hour", "x", 0, 3599)`.

**Bài 3 (điền TODO):** Hoàn thiện `chunk_bounds(start, end, size)` trả về list các cặp `(lo, hi)` của từng khúc.

**Bài 4 (tự viết lại):** Viết lại lớp theo Cách 4 nhưng **chỉ** duyệt từng giây (không có nhánh duyệt khóa), dùng `.get`.

**Bài 5 (mini challenge):** Mở rộng Cách 3 thành lớp `TweetCountsDel` có thêm `deleteTweet(tweetName, time) -> bool` xóa **một** tweet tại đúng thời điểm đó (trả về `False` nếu không có).

In [26]:
# Bài 3 — điền TODO (chỉ định nghĩa, không gọi)
def chunk_bounds(start, end, size):
    res = []
    # TODO: lo bắt đầu từ start, mỗi bước tăng size, hi = min(lo + size - 1, end)
    return res

In [27]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
class TweetCountsSeconds:
    def __init__(self):
        pass  # TODO
    def recordTweet(self, tweetName, time):
        pass  # TODO
    def getTweetCountsPerFrequency(self, freq, tweetName, startTime, endTime):
        pass  # TODO

class TweetCountsDelTodo(TweetCountsBisect):
    def deleteTweet(self, tweetName, time):
        pass  # TODO: bisect_left tìm vị trí, kiểm tra a[i] == time rồi pop

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) `size = 3600`. (2) Số khúc `(10000 − 10)//3600 + 1`. (3) Khúc của `7210` là `(7210 − 10)//3600`.

**Bài 2:** (1) Lọc tweet trong khoảng trước. (2) Khúc phút neo tại 1: `[1,60]`, `[61,120]`, `[121,121]`. (3) Tweet `120` thuộc khúc `(120−1)//60 = 1`.

**Bài 3:** (1) Vòng `while lo <= end`. (2) `hi = min(lo + size − 1, end)`. (3) So độ dài với `(end − start)//size + 1`.

**Bài 4:** (1) `defaultdict(Counter)` hoặc `defaultdict(dict)`. (2) `for x in range(start, end + 1)`. (3) `res[(x − start)//size] += cnt.get(x, 0)`.

**Bài 5:** (1) `i = bisect_left(a, time)`. (2) Nếu `i < len(a) and a[i] == time` thì `a.pop(i)` và trả `True`. (3) Kiểm thử với brute force có thao tác xóa (`list.remove`).

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1
start, end, size = 10, 10000, 3600
print("Bài 1:", (end - start) // size + 1, "khúc; khúc cuối", [10 + 2 * 3600, 10000], "; 7210 thuộc khúc", (7210 - 10) // 3600)
assert (end - start) // size + 1 == 3 and (7210 - 10) // 3600 == 2

# Bài 2
ops = ["TweetCounts"] + ["recordTweet"] * 7 + ["getTweetCountsPerFrequency"] * 2
args = [[]] + [["x", t] for t in [1, 59, 60, 61, 119, 120, 3599]] + [["minute", "x", 1, 121], ["hour", "x", 0, 3599]]
out = run(TweetCounts, ops, args)
print("Bài 2:", out[-2:])
assert out[-2:] == [[3, 3, 0], [7]]

Bài 1: 3 khúc; khúc cuối [7210, 10000] ; 7210 thuộc khúc 2
Bài 2: [[3, 3, 0], [7]]


In [29]:
# Bài 3
def chunk_bounds_sol(start, end, size):
    res, lo = [], start
    while lo <= end:
        res.append((lo, min(lo + size - 1, end)))
        lo += size
    return res

print("Bài 3:", chunk_bounds_sol(10, 200, 60))
assert chunk_bounds_sol(10, 200, 60) == [(10, 69), (70, 129), (130, 189), (190, 200)]
assert chunk_bounds_sol(5, 5, 60) == [(5, 5)]
for s, e, z in [(0, 10**4, 60), (7, 9999, 3600), (3, 3, 86400)]:
    b = chunk_bounds_sol(s, e, z)
    assert len(b) == (e - s) // z + 1 and b[-1][1] == e

# Bài 4
class TweetCountsSecondsSol:
    FREQ = {"minute": 60, "hour": 3600, "day": 86400}
    def __init__(self):
        self.c = defaultdict(dict)
    def recordTweet(self, tweetName, time):
        d = self.c[tweetName]
        d[time] = d.get(time, 0) + 1
    def getTweetCountsPerFrequency(self, freq, tweetName, startTime, endTime):
        size = self.FREQ[freq]
        d = self.c.get(tweetName, {})
        res = [0] * ((endTime - startTime) // size + 1)
        for x in range(startTime, endTime + 1):
            res[(x - startTime) // size] += d.get(x, 0)
        return res

rng = random.Random(4)
for _ in range(60):
    ops, args = random_ops(rng, 40)
    assert run(TweetCountsSecondsSol, ops, args) == run(TweetCountsBrute, ops, args)
print("Bài 3, 4: đúng")

Bài 3: [(10, 69), (70, 129), (130, 189), (190, 200)]


Bài 3, 4: đúng


In [30]:
# Bài 5
class TweetCountsDel(TweetCountsBisect):
    def deleteTweet(self, tweetName, time):
        a = self.t.get(tweetName)
        if not a:
            return False
        i = bisect.bisect_left(a, time)
        if i < len(a) and a[i] == time:
            a.pop(i)
            return True
        return False

class TweetCountsDelBrute(TweetCountsBrute):
    def deleteTweet(self, tweetName, time):
        if (tweetName, time) in self.all:
            self.all.remove((tweetName, time))
            return True
        return False

rng = random.Random(5)
for _ in range(100):
    ops, args = random_ops(rng, 50)
    for _ in range(15):                          # chèn thêm lệnh xóa ở vị trí ngẫu nhiên
        pos = rng.randint(1, len(ops))
        ops.insert(pos, "deleteTweet")
        args.insert(pos, [rng.choice(["t1", "t2"]), rng.choice([a[1] for a in args if len(a) == 2] or [0])])
    assert run(TweetCountsDel, ops, args) == run(TweetCountsDelBrute, ops, args)
print("Bài 5: deleteTweet khớp brute force trên 100 chuỗi lệnh")

Bài 5: deleteTweet khớp brute force trên 100 chuỗi lệnh


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
class TweetCounts:
    def __init__(self):
        self.t = defaultdict(list)
        self.size = {"minute": 60, "hour": 3600, "day": 86400}
    def recordTweet(self, tweetName, time):
        bisect.insort(self.t[tweetName], time)
    def getTweetCountsPerFrequency(self, freq, tweetName, startTime, endTime):
        size, a = self.size[freq], self.t[tweetName]
        res = [0] * ((endTime - startTime) // size + 1)
        for x in a[bisect_left(a, startTime):bisect_right(a, endTime)]:
            res[(x - startTime) // size] += 1
        return res
```

**Checklist trước khi submit:**
- [ ] Số khúc `(end − start)//size + 1` (thử `start == end`).
- [ ] Chỉ số khúc `(t − start)//size`, **trừ** `start`.
- [ ] Lưu cả tweet trùng thời điểm (list/Counter, không set).
- [ ] Cận phải dùng `bisect_right` (khoảng đóng).
- [ ] Tên chưa từng thấy ⇒ trả list toàn 0 đúng độ dài.
- [ ] Đúng tên lớp/phương thức: `TweetCounts`, `recordTweet`, `getTweetCountsPerFrequency`.